# F_S3 — Shape Classification on New Parameterized Data

Implements the full **global-to-shape decision hierarchy**:

1. **Gate 1** — MIC → global relationship strength (strong / intermediate / none)
2. **Gate 2** — |Pearson| & |Spearman| → Simple vs Complex
3. **Simple path, power gate** — power-law fit → Linear / Concave / Convex / next cubic–S–Threshold gate
4. **Complex path** — Branching / Threshold / Turning-point → subtypes

Reads from `F/output/S1_parameterized/` and `F/output/S2_parameterized/`.

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings('ignore')

def count_percent(count, total):
    count = int(count)
    total = int(total)
    return f'{count:,} ({count / total:.1%})' if total else f'{count:,} (N/A)'

def locate_repo_root() -> Path:
    for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (root / 'D').is_dir() and (root / 'E').is_dir():
            return root
    raise FileNotFoundError('Could not locate repo root.')

REPO_ROOT = locate_repo_root()
S1_DIR = REPO_ROOT / 'F' / 'output' / 'S1_parameterized'
S2_DIR = REPO_ROOT / 'F' / 'output' / 'S2_metrics'
OUT_DIR = REPO_ROOT / 'F' / 'output' / 'S3_shape_classification_new_data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input:  {S1_DIR.relative_to(REPO_ROOT)}, {S2_DIR.relative_to(REPO_ROOT)}')
print(f'Output: {OUT_DIR.relative_to(REPO_ROOT)}')

## Family catalogue

Ground-truth properties for each signal family, used for validation at the end.

In [2]:
# Family catalogue: ground-truth properties for each signal family.
# F17 (Quadratic peak) was removed in F; F18 is standalone.
FAMILY_CATALOGUE = {
    'F01': {'name': 'Linear positive',           'direction': 'positive',     'linearity': 'linear',              'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'none',                  'tp': 0},
    'F02': {'name': 'Linear negative',           'direction': 'negative',     'linearity': 'linear',              'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'none',                  'tp': 0},
    'F03': {'name': 'Power convex positive',     'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F04': {'name': 'Power convex negative',     'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F05': {'name': 'Power concave positive',    'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F06': {'name': 'Power concave negative',    'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F07': {'name': 'Saturation positive',       'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'saturation',            'tp': 0},
    'F08': {'name': 'Saturation negative',       'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'saturation',            'tp': 0},
    'F09': {'name': 'Log positive',              'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F10': {'name': 'Log negative',              'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F11': {'name': 'Exponential positive',      'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F12': {'name': 'Exponential negative',      'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F13': {'name': 'S-curve positive',          'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'mixed',  'special_shape': 'S-curve',               'tp': 0},
    'F14': {'name': 'S-curve negative',          'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'mixed',  'special_shape': 'S-curve',               'tp': 0},
    'F15': {'name': 'Threshold positive',        'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'threshold',             'tp': 0},
    'F16': {'name': 'Threshold negative',        'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'threshold',             'tp': 0},
    'F18': {'name': 'Quadratic valley (U)',       'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'convex', 'special_shape': 'valley',                'tp': 1},
    'F19': {'name': 'Spike',                     'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'N/A',    'special_shape': 'spike',                 'tp': 1},
    'F20': {'name': 'Inverted Spike',            'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'N/A',    'special_shape': 'inverted spike',        'tp': 1},
    'F21': {'name': 'Cubic',                     'direction': 'none/mixed',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'mixed',  'special_shape': 'cubic',                 'tp': 2},
    'F22': {'name': 'Oscillation',               'direction': 'none/mixed',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'mixed',  'special_shape': 'oscillation',           'tp': 'multiple'},
    'F23': {'name': 'Two Lines',                 'direction': 'positive',     'linearity': 'multi-branch linear', 'monotonicity': 'multi-branch',  'curvature': 'N/A',    'special_shape': 'two lines',             'tp': 0},
    'F24': {'name': 'Line and Parabola',         'direction': 'none/local',   'linearity': 'multi-branch',        'monotonicity': 'multi-branch',  'curvature': 'mixed',  'special_shape': 'line + parabola',       'tp': 1},
    'F25': {'name': 'Multi-regime threshold',    'direction': 'positive',     'linearity': 'multi-branch',        'monotonicity': 'multi-branch',  'curvature': 'convex', 'special_shape': 'overlapping J',         'tp': 'multiple'},
    'F26': {'name': 'Windowed threshold',        'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'convex', 'special_shape': 'windowed J',            'tp': 'multiple'},
    'Null': {'name': 'No relationship',          'direction': 'N/A',          'linearity': 'N/A',                 'monotonicity': 'N/A',           'curvature': 'N/A',    'special_shape': 'none',                  'tp': 'N/A'},
}

# Build the expected shape_label mapping from the catalogue.
# This is the ground-truth: what shape should the classifier assign at high SNR?
EXPECTED_SHAPE = {}
for fid, props in FAMILY_CATALOGUE.items():
    if fid == 'Null':
        EXPECTED_SHAPE[fid] = 'no_global'
        continue
    d = props['direction']
    mono = props['monotonicity']
    lin = props['linearity']
    curv = props['curvature']
    shape = props['special_shape']
    tp = props['tp']

    # Simple monotonic families → gate2 = Simple
    if mono == 'monotonic':
        if lin == 'linear':
            EXPECTED_SHAPE[fid] = f'{d}_line'
        elif shape == 'S-curve':
            EXPECTED_SHAPE[fid] = f'{d}_s_shaped'
        elif shape == 'threshold':
            EXPECTED_SHAPE[fid] = f'{d}_threshold'
        elif curv == 'concave':
            EXPECTED_SHAPE[fid] = f'{d}_concave'
        elif curv == 'convex':
            EXPECTED_SHAPE[fid] = f'{d}_convex'
        else:
            EXPECTED_SHAPE[fid] = 'simple_other'
    # Non-monotonic / multi-branch → gate2 = Complex
    elif mono == 'non-monotonic':
        if tp == 1:
            EXPECTED_SHAPE[fid] = 'u_inv_u_spike'
        elif tp == 2:
            EXPECTED_SHAPE[fid] = 'complex_cubic'
        elif tp == 'multiple' and 'oscillat' in shape:
            EXPECTED_SHAPE[fid] = 'oscillatory'
        elif tp == 'multiple' and 'windowed' in shape:
            EXPECTED_SHAPE[fid] = 'multi_step_threshold'
        else:
            EXPECTED_SHAPE[fid] = 'complex_uncertain'
    elif mono == 'multi-branch':
        if 'two lines' in shape:
            EXPECTED_SHAPE[fid] = '2_way_branching'
        elif 'parabola' in shape:
            EXPECTED_SHAPE[fid] = '2_way_branching'
        elif 'overlapping' in shape or 'J' in shape:
            EXPECTED_SHAPE[fid] = 'n_way_branching'
        else:
            EXPECTED_SHAPE[fid] = 'complex_uncertain'

catalogue_df = pd.DataFrame.from_dict(FAMILY_CATALOGUE, orient='index').reset_index(names='family_id')
catalogue_df['expected_shape'] = catalogue_df['family_id'].map(EXPECTED_SHAPE)
print(f'{len(FAMILY_CATALOGUE)} families loaded')
display(catalogue_df[['family_id', 'name', 'monotonicity', 'curvature', 'special_shape', 'expected_shape']])

26 families loaded


,family_id,name,monotonicity,curvature,special_shape,expected_shape
0,F01,Linear positive,monotonic,N/A,none,positive_line
1,F02,Linear negative,monotonic,N/A,none,negative_line
2,F03,Power convex positive,monotonic,convex,none,positive_convex
3,F04,Power convex negative,monotonic,concave,none,negative_concave
4,F05,Power concave positive,monotonic,concave,none,positive_concave
5,F06,Power concave negative,monotonic,convex,none,negative_convex
6,F07,Saturation positive,monotonic,concave,saturation,positive_concave
7,F08,Saturation negative,monotonic,convex,saturation,negative_convex
8,F09,Log positive,monotonic,concave,none,positive_concave
9,F10,Log negative,monotonic,convex,none,negative_convex


## Load data

In [ ]:
# ── Strength level filter ──
# Set to a single level or a list to analyse only those levels.
# Set to None to use all levels (no filtering).
# Available: 'very_weak', 'weak', 'medium', 'strong', 'very_strong', 'not_applicable'
STRENGTH_FILTER = 'strong'        # e.g. 'strong', ['strong', 'very_strong'], or None

cases_df = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_df['_pt_idx'] = np.arange(len(cases_df), dtype=np.int64)
metrics_df = pd.read_parquet(S2_DIR / 'metrics_full.parquet')
pts = np.load(S1_DIR / 'scatter_points.npz')
x_all = pts['x']
y_all = pts['y']
del pts

if len(x_all) != len(cases_df) or len(y_all) != len(cases_df):
    raise ValueError('Scatter arrays and cases.csv must contain the same number of rows.')
if cases_df['case_id'].duplicated().any() or metrics_df['case_id'].duplicated().any():
    raise ValueError('case_id must be unique in both cases and metrics.')

# Apply strength filter before merge (keeps _pt_idx pointing into original x_all/y_all)
if STRENGTH_FILTER is not None:
    levels = [STRENGTH_FILTER] if isinstance(STRENGTH_FILTER, str) else list(STRENGTH_FILTER)
    _before = len(cases_df)
    cases_df = cases_df[cases_df['strength_name'].isin(levels)].reset_index(drop=True)
    print(f'Strength filter: {levels} → kept {len(cases_df):,} / {_before:,} cases')
else:
    print('Strength filter: OFF (all levels)')

data = cases_df.merge(metrics_df, on='case_id', how='inner', validate='one_to_one')
data = data.reset_index(drop=True)
data['is_null'] = data['family_id'].eq('Null')
data['_pt_idx'] = pd.to_numeric(data['_pt_idx'], errors='raise').astype(np.int64)
if not data['_pt_idx'].between(0, len(x_all) - 1).all():
    raise IndexError('A mapped scatter-point index is outside the stored arrays.')
if data['_pt_idx'].duplicated().any():
    raise ValueError('Multiple classified cases map to the same scatter-point row.')

print(f'Loaded {len(data):,} metric-covered cases × {x_all.shape[1]} points')
print(f'  Metrics coverage: {len(data):,} / {len(cases_df):,} ({len(data) / len(cases_df):.1%})')
print(f'  Signal: {(~data["is_null"]).sum():,}   Null: {data["is_null"].sum():,}')
print(f'  MIC range: [{data["MIC"].min():.3f}, {data["MIC"].max():.3f}]')

## Gate 1 — Global relationship (MIC)

- **Strong**: MIC ≥ 0.8 → proceed to Gate 2
- **Intermediate**: 0.6 ≤ MIC < 0.8 → classification stops here
- **No global**: MIC < 0.6 → no relationship detected

In [ ]:
# ── Gate 1 thresholds ──
MIC_STRONG = 0.8
MIC_INTERMEDIATE = 0.6

def assign_gate1(mic):
    if mic >= MIC_STRONG:
        return 'strong_global'
    elif mic >= MIC_INTERMEDIATE:
        return 'intermediate_global'
    else:
        return 'no_global'

data['gate1'] = data['MIC'].apply(assign_gate1)

_g1_counts = data.groupby(['family_id', 'gate1']).size().unstack(fill_value=0)
_g1_counts = _g1_counts.reindex(
    columns=['strong_global', 'intermediate_global', 'no_global'], fill_value=0
).sort_index()
_g1_totals = _g1_counts.sum(axis=1)
_g1 = pd.DataFrame(index=_g1_counts.index)
_g1['family_name'] = _g1.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
_g1['Total'] = [count_percent(n, n) for n in _g1_totals]
for col in _g1_counts.columns:
    _g1[col] = [count_percent(n, total) for n, total in zip(_g1_counts[col], _g1_totals)]
print(f'=== Gate 1: MIC >= {MIC_STRONG} → strong, >= {MIC_INTERMEDIATE} → intermediate ===')
display(_g1)
print(f'\nOverall: {count_percent((data["gate1"]=="strong_global").sum(), len(data))} strong / '
      f'{count_percent((data["gate1"]=="intermediate_global").sum(), len(data))} intermediate / '
      f'{count_percent((data["gate1"]=="no_global").sum(), len(data))} no_global')

## Gate 2 — Simple or Complex

Select MIC range with `MIC_ANALYSIS_GATE`:
- `'strong_global'` → MIC ≥ 0.8
- `'intermediate_global'` → 0.6 ≤ MIC < 0.8

Then apply correlation filter to determine Simple vs Complex.

In [ ]:
# ── Gate 2 thresholds ──
PEARSON_THRESHOLD = 0.7
USE_SPEARMAN_GATE = False
SPEARMAN_THRESHOLD = 0.7

# ── MIC range selector ──
# MIC_ANALYSIS_GATE = 'strong_global'          # MIC >= 0.8
MIC_ANALYSIS_GATE = 'intermediate_global'      # 0.6 <= MIC < 0.8

mask_analysis = data['gate1'] == MIC_ANALYSIS_GATE
data['gate2'] = None

pearson_ok = data.loc[mask_analysis, 'pearson_r'].abs() >= PEARSON_THRESHOLD
if USE_SPEARMAN_GATE:
    spearman_ok = data.loc[mask_analysis, 'spearman_rho'].abs() >= SPEARMAN_THRESHOLD
    simple_mask = pearson_ok & spearman_ok
    gate2_desc = f'Simple = |Pearson| >= {PEARSON_THRESHOLD} AND |Spearman| >= {SPEARMAN_THRESHOLD}'
else:
    simple_mask = pearson_ok
    gate2_desc = f'Simple = |Pearson| >= {PEARSON_THRESHOLD} (Spearman gate OFF)'

data.loc[mask_analysis, 'gate2'] = np.where(simple_mask, 'Simple', 'Complex')

_g2_data = data.loc[mask_analysis]
_g2_counts = _g2_data.groupby(['family_id', 'gate2']).size().unstack(fill_value=0)
_g2_counts = _g2_counts.reindex(columns=['Simple', 'Complex'], fill_value=0).sort_index()
_g2_totals = _g2_counts.sum(axis=1)
_g2 = pd.DataFrame(index=_g2_counts.index)
_g2['family_name'] = _g2.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
_g2['Total'] = [count_percent(n, n) for n in _g2_totals]
for col in _g2_counts.columns:
    _g2[col] = [count_percent(n, total) for n, total in zip(_g2_counts[col], _g2_totals)]
print(f'=== Gate 2 [{MIC_ANALYSIS_GATE}]: {gate2_desc} ===')
display(_g2)
print(f'\nOverall: {count_percent(_g2_data["gate2"].eq("Simple").sum(), len(_g2_data))} Simple / '
      f'{count_percent(_g2_data["gate2"].eq("Complex").sum(), len(_g2_data))} Complex '
      f'out of {len(_g2_data):,} {MIC_ANALYSIS_GATE}')

## Simple path — Power-law fit

This block performs only the first Simple-path layer:

- Fit $y=ax^b+c$.
- **Mode A** (`CLASSIFICATION_ORDER = 'r2_first'`, default): Check $R^2 \geq$ threshold first, then classify by $|b-1|$ and curvature.
- **Mode B** (`CLASSIFICATION_ORDER = 'linear_first'`): Check $|b-1| \leq$ tolerance first to extract linear, then use $R^2$ gate for concave/convex.
- A failed or insufficient power-law fit is passed to the next Cubic/S/Threshold block.

In [ ]:
# ── Simple path thresholds ──
R2_POWER_THRESHOLD = 0.85
LINEAR_B_TOLERANCE = 0.05

# ── Classification order switch ──
# 'r2_first'     : (default) check R² >= threshold, then check |b-1| for linear
# 'linear_first' : check |b-1| <= tolerance first to extract linear, then R² for concave/convex
CLASSIFICATION_ORDER = 'r2_first'   # switch: 'r2_first' or 'linear_first'

def _power_law(x, a, b, c):
    return a * np.power(np.maximum(x, 1e-10), b) + c


def _fit_power_law(x, y):
    valid = np.isfinite(x) & np.isfinite(y) & (x > 1e-6)
    if valid.sum() < 10:
        return None
    x, y = x[valid], y[valid]
    slope = np.polyfit(x, y, 1)[0]
    a0 = slope
    for b0 in [1.0, 0.5, 2.0, 0.3, 3.0]:
        try:
            popt, _ = curve_fit(
                _power_law, x, y, p0=[a0, b0, np.median(y)],
                maxfev=5000,
                bounds=([-np.inf, 0.05, -np.inf], [np.inf, 8.0, np.inf]),
            )
            y_pred = _power_law(x, *popt)
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - y.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            if r2 >= 0:
                return (*popt, r2)
        except (RuntimeError, ValueError):
            continue
    return None


def classify_power_law_stage(x, y):
    """Return a terminal power-law label, or None for cubic fallthrough.

    Two modes controlled by CLASSIFICATION_ORDER:
      'r2_first'     – R² gate first, then |b-1| for linear  (default)
      'linear_first' – |b-1| gate first for linear, then R² for concave/convex
    """
    pw = _fit_power_law(x, y)
    if pw is None:
        return None

    a, b, c, r2 = pw
    ab = a * b
    direction = 'positive' if ab > 0 else 'negative'
    details = {'step': 'power_law', 'a': a, 'b': b, 'c': c, 'r2': r2}

    if CLASSIFICATION_ORDER == 'linear_first':
        # Mode B: check |b-1| first → linear; then R² for concave/convex
        if abs(b - 1) <= LINEAR_B_TOLERANCE:
            return f'{direction}_line', details
        if r2 >= R2_POWER_THRESHOLD:
            curvature = a * b * (b - 1)
            if curvature < 0:
                return f'{direction}_concave', details
            else:
                return f'{direction}_convex', details
        return None
    else:
        # Mode A (default): R² gate first, then classify
        if r2 >= R2_POWER_THRESHOLD:
            if abs(b - 1) <= LINEAR_B_TOLERANCE:
                return f'{direction}_line', details
            curvature = a * b * (b - 1)
            if curvature < 0:
                return f'{direction}_concave', details
            else:
                return f'{direction}_convex', details
        return None

print(f'Power-law stage helpers defined.  CLASSIFICATION_ORDER = {CLASSIFICATION_ORDER!r}')

## Simple path — Cubic polynomial fit

This is the next layer and receives only cases that were not resolved by the power-law block.

- Fit $f(x)=a_3x^3+a_2x^2+a_1x+a_0$.
- Calculate the roots of $f'(x)=3a_3x^2+2a_2x+a_1$ in the central domain.
- At least two interior turning points → **Cubic**.
- Otherwise → combined **S/Threshold** class.

In [ ]:
# ── Cubic polynomial fit thresholds ──
CUBIC_TP_DOMAIN = (0.05, 0.95)
CUBIC_MIN_TURNING_POINTS = 2

def _fit_cubic(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        return None
    x, y = x[valid], y[valid]
    try:
        coeffs = np.polyfit(x, y, 3)
        y_pred = np.polyval(coeffs, x)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return coeffs, r2
    except (np.RankWarning, np.linalg.LinAlgError):
        return None


def _count_turning_points(coeffs):
    """Count real roots of f'(x) inside CUBIC_TP_DOMAIN."""
    roots = np.roots(np.polyder(coeffs))
    real_roots = roots[np.isreal(roots)].real
    lower, upper = CUBIC_TP_DOMAIN
    interior = real_roots[(real_roots >= lower) & (real_roots <= upper)]
    return len(interior), np.sort(interior)


def classify_cubic_stage(x, y):
    """Classify a power-law fallthrough as Cubic, S/Threshold, or Other."""
    cb = _fit_cubic(x, y)
    if cb is None:
        return 'simple_other', {'step': 'failed'}

    coeffs, r2_cubic = cb
    n_tp, tp_locs = _count_turning_points(coeffs)
    direction_val = float(np.polyval(coeffs, 1.0) - np.polyval(coeffs, 0.0))
    direction = 'positive' if direction_val >= 0 else 'negative'
    details = {
        'step': 'cubic', 'coeffs': coeffs.tolist(),
        'r2': r2_cubic, 'n_tp': n_tp, 'tp_locs': tp_locs.tolist(),
    }

    if n_tp >= CUBIC_MIN_TURNING_POINTS:
        return 'cubic_two_turning_point', details
    return f'{direction}_s_threshold', details


def classify_simple_path(x, y):
    """Run the two sequential Simple-path blocks."""
    power_result = classify_power_law_stage(x, y)
    if power_result is not None:
        return power_result
    return classify_cubic_stage(x, y)

print('Cubic polynomial fit helpers defined.')

## Run full classification

In [ ]:
import time

pt_idx = data['_pt_idx'].values
shape_labels = np.full(len(data), '', dtype=object)
shape_details = np.full(len(data), '', dtype=object)

# Cases outside analysis range
mask_outside = ~mask_analysis
shape_labels[mask_outside.values] = data.loc[mask_outside, 'gate1']
shape_details[mask_outside.values] = '{}'

mask_simple = (data['gate2'] == 'Simple').values
mask_complex = (data['gate2'] == 'Complex').values
idx_simple = np.where(mask_simple)[0]

print(f'MIC range: {MIC_ANALYSIS_GATE}')
print(f'To classify: {len(idx_simple):,} Simple')
print(f'Complex (not sub-classified): {mask_complex.sum():,}')

t0 = time.time()
for i in tqdm(idx_simple, desc='Simple path'):
    pi = pt_idx[i]
    x = x_all[pi].astype(np.float64)
    y = y_all[pi].astype(np.float64)
    label, details = classify_simple_path(x, y)
    shape_labels[i] = label
    shape_details[i] = json.dumps(details, default=str)
print(f'Simple path: {len(idx_simple):,} cases in {time.time() - t0:.1f}s')

shape_labels[mask_complex] = 'complex'
shape_details[mask_complex] = '{}'

data['shape_label'] = shape_labels
data['shape_details'] = shape_details

# Parse shape_details for step-by-step analysis
_det = data.loc[mask_simple, 'shape_details'].apply(
    lambda s: json.loads(s) if isinstance(s, str) and s != '{}' else {}
)
data.loc[mask_simple, '_step'] = _det.map(lambda d: d.get('step', ''))
data.loc[mask_simple, '_r2'] = _det.map(lambda d: d.get('r2', np.nan))
data.loc[mask_simple, '_b'] = _det.map(lambda d: d.get('b', np.nan))
data.loc[mask_simple, '_a'] = _det.map(lambda d: d.get('a', np.nan))

SHAPE_TO_BROAD = {
    'positive_line': 'line', 'negative_line': 'line',
    'positive_concave': 'concave', 'negative_concave': 'concave',
    'positive_convex': 'convex', 'negative_convex': 'convex',
    'cubic_two_turning_point': 'cubic',
    'positive_s_threshold': 'S/threshold', 'negative_s_threshold': 'S/threshold',
    'simple_other': 'other',
}
data['broad_shape'] = data['shape_label'].map(SHAPE_TO_BROAD)

# ── Expected power-law route per family (ground truth) ──
EXPECTED_POWER_ROUTE = {}
for _fid, _exp in EXPECTED_SHAPE.items():
    if _exp.endswith('_line'):
        EXPECTED_POWER_ROUTE[_fid] = 'Linear'
    elif _exp.endswith('_concave'):
        EXPECTED_POWER_ROUTE[_fid] = 'Concave'
    elif _exp.endswith('_convex'):
        EXPECTED_POWER_ROUTE[_fid] = 'Convex'
    else:
        EXPECTED_POWER_ROUTE[_fid] = 'Enter next branch'

# ────────────────────────────────────────────
# Table 1: Simple / Complex entry
# ────────────────────────────────────────────
print(f'\n=== Table 1: Simple entry ({MIC_ANALYSIS_GATE} → Simple / Complex) ===')
_t1 = data[mask_analysis].copy()
_t1_counts = _t1.groupby(['family_id', 'gate2']).size().unstack(fill_value=0)
_t1_counts = _t1_counts.reindex(columns=['Simple', 'Complex'], fill_value=0)
_t1_counts['total'] = _t1_counts.sum(axis=1)
_t1_out = pd.DataFrame(index=_t1_counts.index)
_t1_out['family_name'] = _t1_out.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
_t1_out['total'] = [count_percent(n, n) for n in _t1_counts['total']]
for col in ['Simple', 'Complex']:
    _t1_out[col] = [count_percent(n, t) for n, t in zip(_t1_counts[col], _t1_counts['total'])]
display(_t1_out)

# ────────────────────────────────────────────
# Table 2: Power-law R² >= threshold?
# ────────────────────────────────────────────
_all_simple = data.loc[mask_simple].copy()
_all_simple['r2_pass'] = (data.loc[mask_simple, '_step'] == 'power_law')
print(f'\n=== Table 2: Power-law R² >= {R2_POWER_THRESHOLD} (all Simple) ===')
_t2 = _all_simple.groupby('family_id').agg(
    total_simple=('case_id', 'size'),
    r2_pass=('r2_pass', 'sum'),
).assign(r2_fail=lambda d: d['total_simple'] - d['r2_pass'])
_t2_out = pd.DataFrame(index=_t2.index)
_t2_out['family_name'] = _t2_out.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
_t2_out['total_simple'] = [count_percent(n, n) for n in _t2['total_simple']]
_t2_out[f'R²≥{R2_POWER_THRESHOLD}'] = [count_percent(n, t) for n, t in zip(_t2['r2_pass'], _t2['total_simple'])]
_t2_out[f'R²<{R2_POWER_THRESHOLD}'] = [count_percent(n, t) for n, t in zip(_t2['r2_fail'], _t2['total_simple'])]
display(_t2_out)

# ────────────────────────────────────────────
# Table 3: |b-1| <= tolerance? (among R² pass)
# ────────────────────────────────────────────
_pw = _all_simple[_all_simple['r2_pass']].copy()
_pw['_b'] = data.loc[_pw.index, '_b']
_pw['_a'] = data.loc[_pw.index, '_a']
_pw['is_linear'] = (_pw['_b'] - 1).abs() <= LINEAR_B_TOLERANCE
print(f'\n=== Table 3: Linear |b−1| <= {LINEAR_B_TOLERANCE} (among R² pass) ===')
_t3 = _pw.groupby('family_id').agg(
    r2_pass=('case_id', 'size'),
    linear=('is_linear', 'sum'),
).assign(not_linear=lambda d: d['r2_pass'] - d['linear'])
_t3_out = pd.DataFrame(index=_t3.index)
_t3_out['family_name'] = _t3_out.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
_t3_out['R²_pass'] = [count_percent(n, n) for n in _t3['r2_pass']]
_t3_out['Linear'] = [count_percent(n, t) for n, t in zip(_t3['linear'], _t3['r2_pass'])]
_t3_out['Not linear (→curvature)'] = [count_percent(n, t) for n, t in zip(_t3['not_linear'], _t3['r2_pass'])]
display(_t3_out)

# ────────────────────────────────────────────
# Table 4: Curvature sign (among not-linear R² pass)
# ────────────────────────────────────────────
_curv = _pw[~_pw['is_linear']].copy()
_curv['curvature_sign'] = np.sign(_curv['_a'] * _curv['_b'] * (_curv['_b'] - 1))
_curv['curv_label'] = _curv['curvature_sign'].map({-1.0: 'Concave', 1.0: 'Convex', 0.0: 'Zero'}).fillna('Unknown')
print('\n=== Table 4: Curvature a·b·(b−1) (among not-linear R² pass) ===')
if len(_curv) > 0:
    _t4 = _curv.groupby(['family_id', 'curv_label']).size().unstack(fill_value=0)
    for c in ['Concave', 'Convex']:
        if c not in _t4.columns:
            _t4[c] = 0
    _t4['total_not_linear'] = _t4.sum(axis=1)
    _t4_out = pd.DataFrame(index=_t4.index)
    _t4_out['family_name'] = _t4_out.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
    _t4_out['total_not_linear'] = [count_percent(n, n) for n in _t4['total_not_linear']]
    for c in ['Concave', 'Convex']:
        _t4_out[c] = [count_percent(n, t) for n, t in zip(_t4[c], _t4['total_not_linear'])]
    display(_t4_out)
else:
    print('  No cases in this stage.')

# ────────────────────────────────────────────
# Table 5: Overall power-law routing + Accuracy
# ────────────────────────────────────────────
POWER_ROUTE_ORDER = ['Linear', 'Concave', 'Convex', 'Enter next branch']
_all_simple['power_route'] = _all_simple['broad_shape'].map({
    'line': 'Linear', 'concave': 'Concave', 'convex': 'Convex',
}).fillna('Enter next branch')

power_counts = _all_simple.groupby(['family_id', 'power_route']).size().unstack(fill_value=0)
power_counts = power_counts.reindex(columns=POWER_ROUTE_ORDER, fill_value=0).sort_index()
power_totals = power_counts.sum(axis=1)

power_table = pd.DataFrame(index=power_counts.index)
power_table['family_name'] = power_table.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
power_table['Total'] = [count_percent(n, n) for n in power_totals]
for route in POWER_ROUTE_ORDER:
    power_table[route] = [count_percent(n, t) for n, t in zip(power_counts[route], power_totals)]

# Accuracy: correct route / total
_correct = pd.Series(0, index=power_counts.index, dtype=int)
for fid in power_counts.index:
    expected = EXPECTED_POWER_ROUTE.get(fid, 'Enter next branch')
    if expected in power_counts.columns:
        _correct[fid] = int(power_counts.loc[fid, expected])
power_table['Accuracy'] = [f'{c:,}/{t:,} ({c/t:.1%})' if t else 'N/A'
                           for c, t in zip(_correct, power_totals)]

# Overall row
_overall_counts = power_counts.sum()
_overall_total = int(power_totals.sum())
_overall_correct = int(_correct.sum())
_overall = pd.DataFrame([{
    'family_name': 'All families',
    'Total': count_percent(_overall_total, _overall_total),
    **{r: count_percent(int(_overall_counts[r]), _overall_total) for r in POWER_ROUTE_ORDER},
    'Accuracy': f'{_overall_correct:,}/{_overall_total:,} ({_overall_correct/_overall_total:.1%})'
    if _overall_total else 'N/A',
}], index=['Overall'])
power_table = pd.concat([power_table, _overall])

print(f'\n=== Table 5: Overall power-law routing ({MIC_ANALYSIS_GATE}, R²≥{R2_POWER_THRESHOLD}) ===')
display(power_table)

In [ ]:
# ── R² threshold sweep: power-law accuracy at different thresholds ──
# Pre-compute power-law R², a, b for ALL Simple cases (once)
_simple_data = data.loc[mask_simple].copy()
_pw_r2 = np.full(len(_simple_data), np.nan)
_pw_a = np.full(len(_simple_data), np.nan)
_pw_b = np.full(len(_simple_data), np.nan)

for j, (idx, row) in enumerate(tqdm(_simple_data.iterrows(), total=len(_simple_data), desc='Power-law fits')):
    pi = int(row['_pt_idx'])
    x = x_all[pi].astype(np.float64)
    y = y_all[pi].astype(np.float64)
    pw = _fit_power_law(x, y)
    if pw is not None:
        _pw_a[j], _pw_b[j], _, _pw_r2[j] = pw

_simple_data = _simple_data.assign(pw_r2=_pw_r2, pw_a=_pw_a, pw_b=_pw_b)

# Sweep thresholds
R2_THRESHOLDS = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.98]

def _route_at_threshold(row, threshold):
    if CLASSIFICATION_ORDER == 'linear_first':
        # Mode B: |b-1| first for linear, then R² for concave/convex
        if not np.isnan(row['pw_b']) and abs(row['pw_b'] - 1) <= LINEAR_B_TOLERANCE:
            return 'Linear'
        if np.isnan(row['pw_r2']) or row['pw_r2'] < threshold:
            return 'Enter next branch'
        curv = row['pw_a'] * row['pw_b'] * (row['pw_b'] - 1)
        return 'Concave' if curv < 0 else 'Convex'
    else:
        # Mode A (default): R² first, then classify
        if np.isnan(row['pw_r2']) or row['pw_r2'] < threshold:
            return 'Enter next branch'
        b = row['pw_b']
        a = row['pw_a']
        if abs(b - 1) <= LINEAR_B_TOLERANCE:
            return 'Linear'
        curv = a * b * (b - 1)
        return 'Concave' if curv < 0 else 'Convex'

rows = []
for thr in R2_THRESHOLDS:
    _simple_data['_route'] = _simple_data.apply(lambda r: _route_at_threshold(r, thr), axis=1)
    _simple_data['_expected'] = _simple_data['family_id'].map(EXPECTED_POWER_ROUTE).fillna('Enter next branch')
    _simple_data['_correct'] = _simple_data['_route'] == _simple_data['_expected']

    per_family = _simple_data.groupby('family_id').agg(
        total=('case_id', 'size'),
        correct=('_correct', 'sum'),
    )
    per_family['accuracy'] = per_family['correct'] / per_family['total']

    row_data = {'R² threshold': thr}
    for fid in sorted(per_family.index):
        fname = FAMILY_CATALOGUE.get(fid, {}).get('name', fid)
        n_correct = int(per_family.loc[fid, 'correct'])
        n_total = int(per_family.loc[fid, 'total'])
        row_data[fname] = f'{n_correct}/{n_total} ({n_correct/n_total:.1%})'
    overall_correct = int(per_family['correct'].sum())
    overall_total = int(per_family['total'].sum())
    row_data['Overall'] = f'{overall_correct}/{overall_total} ({overall_correct/overall_total:.1%})'
    rows.append(row_data)

sweep_df = pd.DataFrame(rows).set_index('R² threshold')
print(f'=== Power-law accuracy at different R² thresholds ({MIC_ANALYSIS_GATE}, mode={CLASSIFICATION_ORDER}) ===')
display(sweep_df.T)

### Cubic polynomial fit — $R^2$ diagnostic table

This table reports cubic-polynomial fit quality only for cases that reached the cubic block after the power-law stage. It is diagnostic only and does not alter the turning-point classification rule.

In [ ]:
# # ── Cubic polynomial R² reporting threshold ──
# CUBIC_R2_REPORT_THRESHOLD = 0.90

# _cubic_input = data.loc[
#     mask_simple & data['_step'].isin(['cubic', 'failed'])
# ].copy()
# _cubic_input['cubic_r2_pass'] = (
#     _cubic_input['_step'].eq('cubic')
#     & _cubic_input['_r2'].ge(CUBIC_R2_REPORT_THRESHOLD)
# )

# _cr2_counts = _cubic_input.groupby('family_id').agg(
#     cubic_input=('case_id', 'size'),
#     cubic_r2_pass=('cubic_r2_pass', 'sum'),
# ).reindex(ROUTING_FAMILIES, fill_value=0)
# _cr2_counts['cubic_r2_below_or_failed'] = (
#     _cr2_counts['cubic_input'] - _cr2_counts['cubic_r2_pass']
# )

# _cr2_table = pd.DataFrame(index=_cr2_counts.index)
# _cr2_table['family_name'] = _cr2_table.index.map(
#     lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', '')
# )
# _cr2_table['Cubic polynomial input'] = [
#     count_percent(n, n) if n else '0 (N/A)'
#     for n in _cr2_counts['cubic_input']
# ]
# _cr2_table[f'Cubic R²≥{CUBIC_R2_REPORT_THRESHOLD}'] = [
#     count_percent(n, total) if total else '0 (N/A)'
#     for n, total in zip(_cr2_counts['cubic_r2_pass'], _cr2_counts['cubic_input'])
# ]
# _cr2_table[f'Cubic R²<{CUBIC_R2_REPORT_THRESHOLD} / failed'] = [
#     count_percent(n, total) if total else '0 (N/A)'
#     for n, total in zip(
#         _cr2_counts['cubic_r2_below_or_failed'], _cr2_counts['cubic_input']
#     )
# ]

# print(
#     f'=== Cubic polynomial fit quality (cases entering cubic block; '
#     f'diagnostic threshold R²={CUBIC_R2_REPORT_THRESHOLD}) ==='
# )
# display(_cr2_table)

# _cr2_counts.reset_index().to_csv(
#     OUT_DIR / 'cubic_polynomial_r2_diagnostic.csv', index=False
# )

# # ── Exactly two interior turning points (TP = 2) ──
# _cubic_input['tp_equals_2'] = _cubic_input['_n_tp'].eq(2)
# _tp2_counts = _cubic_input.groupby('family_id').agg(
#     cubic_input=('case_id', 'size'),
#     tp_equals_2=('tp_equals_2', 'sum'),
# ).reindex(ROUTING_FAMILIES, fill_value=0)
# _tp2_counts['tp_not_2'] = (
#     _tp2_counts['cubic_input'] - _tp2_counts['tp_equals_2']
# )

# _tp2_table = pd.DataFrame(index=_tp2_counts.index)
# _tp2_table['family_name'] = _tp2_table.index.map(
#     lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', '')
# )
# _tp2_table['Cubic polynomial input'] = [
#     count_percent(n, n) if n else '0 (N/A)'
#     for n in _tp2_counts['cubic_input']
# ]
# _tp2_table['TP = 2'] = [
#     count_percent(n, total) if total else '0 (N/A)'
#     for n, total in zip(_tp2_counts['tp_equals_2'], _tp2_counts['cubic_input'])
# ]
# _tp2_table['TP ≠ 2'] = [
#     count_percent(n, total) if total else '0 (N/A)'
#     for n, total in zip(_tp2_counts['tp_not_2'], _tp2_counts['cubic_input'])
# ]

# print('\n=== Cubic polynomial turning-point diagnostic: TP = 2 ===')
# display(_tp2_table)

# _tp2_counts.reset_index().to_csv(
#     OUT_DIR / 'cubic_polynomial_tp2_diagnostic.csv', index=False
# )

## Save results

In [ ]:
output_cols = [
    'case_id', '_pt_idx', 'family_id', 'family_name', 'canonical_family',
    'strength_name', 'snr', 'snr_float', 'replicate',
    'MIC', 'pearson_r', 'spearman_rho',
    'gate1', 'gate2', 'shape_label', 'broad_shape', 'shape_details',
]
if 'is_representative' in data.columns:
    output_cols.insert(4, 'is_representative')

result = data[[c for c in output_cols if c in data.columns]].copy()
result = result.rename(columns={'_pt_idx': 'scatter_point_index'})
result.to_parquet(OUT_DIR / 'shape_classification.parquet', index=False)
result.to_csv(OUT_DIR / 'shape_classification.csv', index=False)

print(f'Saved {len(result):,} rows to {OUT_DIR.relative_to(REPO_ROOT)}/')

## Summary & validation

Compare predicted shape labels against the known `family_id` ground truth.

In [ ]:
# Overall shape label distribution
print('=== Shape label distribution ===')
_shape_counts = data['shape_label'].value_counts()
_shape_distribution = pd.DataFrame({
    'shape_label': _shape_counts.index,
    'Count (share)': [count_percent(n, len(data)) for n in _shape_counts],
})
print(_shape_distribution.to_string(index=False))
print()

# Evaluate the current Simple-path label resolution rather than comparing it
# with finer Complex labels that this notebook does not attempt to predict.
EXPECTED_BROAD_SHAPE = {
    'F01': 'line', 'F02': 'line',
    'F03': 'convex', 'F04': 'concave',
    'F05': 'concave', 'F06': 'convex',
    'F07': 'concave', 'F08': 'convex',
    'F09': 'concave', 'F10': 'convex',
    'F11': 'convex', 'F12': 'concave',
    'F13': 'S/threshold', 'F14': 'S/threshold',
    'F15': 'S/threshold', 'F16': 'S/threshold',
    'F21': 'cubic',
}

high_snr = data[pd.to_numeric(data['snr_float'], errors='coerce') >= 10].copy()
high_snr['expected_broad_shape'] = high_snr['family_id'].map(EXPECTED_BROAD_SHAPE)
eligible_high = high_snr[high_snr['expected_broad_shape'].notna()].copy()
eligible_high['entered_simple'] = eligible_high['gate2'].eq('Simple')
shape_eval = eligible_high[eligible_high['entered_simple']].copy()
shape_eval['predicted_broad_shape'] = shape_eval['broad_shape'].fillna('other')
shape_eval['correct'] = (
    shape_eval['predicted_broad_shape'] == shape_eval['expected_broad_shape']
)

coverage = eligible_high.groupby('family_id', as_index=False).agg(
    family_name=('family_name', 'first'),
    expected_broad_shape=('expected_broad_shape', 'first'),
    high_snr_input=('case_id', 'size'),
    entered_simple=('entered_simple', 'sum'),
)
coverage['simple_path_coverage'] = coverage['entered_simple'] / coverage['high_snr_input']
shape_accuracy = shape_eval.groupby('family_id', as_index=False).agg(
    correctly_classified=('correct', 'sum'),
    top_predicted=('predicted_broad_shape', lambda s: s.value_counts().index[0]),
)
accuracy = coverage.merge(shape_accuracy, on='family_id', how='left')
accuracy['correctly_classified'] = accuracy['correctly_classified'].fillna(0).astype(int)
accuracy['conditional_accuracy'] = np.divide(
    accuracy['correctly_classified'], accuracy['entered_simple'],
    out=np.full(len(accuracy), np.nan), where=accuracy['entered_simple'].to_numpy() > 0,
)

print('=== High-SNR Simple-path coverage and broad-shape accuracy ===')
accuracy_display = accuracy[[
    'family_id', 'family_name', 'expected_broad_shape', 'high_snr_input',
    'entered_simple', 'correctly_classified', 'top_predicted',
]].copy()
accuracy_display['High-SNR input'] = [count_percent(n, n) for n in accuracy['high_snr_input']]
accuracy_display['Entered Simple'] = [
    count_percent(n, total) for n, total in zip(accuracy['entered_simple'], accuracy['high_snr_input'])
]
accuracy_display['Correct'] = [
    count_percent(n, total) for n, total in zip(accuracy['correctly_classified'], accuracy['entered_simple'])
]
accuracy_display = accuracy_display[[
    'family_id', 'family_name', 'expected_broad_shape', 'High-SNR input',
    'Entered Simple', 'top_predicted', 'Correct',
]]
print(accuracy_display.to_string(index=False))
overall_coverage = len(shape_eval) / len(eligible_high) if len(eligible_high) else np.nan
overall_accuracy = shape_eval['correct'].mean() if len(shape_eval) else np.nan
print(f'\nSimple-path coverage: {len(shape_eval):,}/{len(eligible_high):,} ({overall_coverage:.1%})')
print(f'Conditional broad-shape accuracy: {shape_eval["correct"].sum():,}/{len(shape_eval):,} ({overall_accuracy:.1%})')
print('Complex families are excluded here because this notebook currently reports only the broad label Complex.')
accuracy.to_csv(OUT_DIR / 'simple_path_broad_accuracy_high_snr.csv', index=False)

In [10]:
# Cross-tabulation: family × shape_label for high SNR
print('=== Family × Shape Label (SNR >= 10, strong_global only) ===')
ct_counts = high_snr[high_snr['gate1'] == 'strong_global'].groupby(
    ['family_id', 'shape_label']
).size().unstack(fill_value=0)
ct_totals = ct_counts.sum(axis=1)
ct = pd.DataFrame(index=ct_counts.index)
ct['family_name'] = ct.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
ct['Total'] = [count_percent(n, n) for n in ct_totals]
for col in ct_counts.columns:
    ct[col] = [count_percent(n, total) for n, total in zip(ct_counts[col], ct_totals)]
display(ct)

=== Family × Shape Label (SNR >= 10, strong_global only) ===


,family_name,Total,complex,cubic_two_turning_point,negative_concave,negative_convex,negative_s_threshold,positive_concave,positive_convex,positive_line,positive_s_threshold
family_id,,,,,,,,,,,
F01,Linear positive,"10,440 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),134 (1.3%),151 (1.4%),"10,150 (97.2%)",5 (0.0%)
F03,Power convex positive,"10,094 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),"10,089 (100.0%)",0 (0.0%),5 (0.0%)
F05,Power concave positive,"10,404 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),"10,395 (99.9%)",0 (0.0%),0 (0.0%),9 (0.1%)
F07,Saturation positive,"2,936 (100.0%)",758 (25.8%),767 (26.1%),0 (0.0%),0 (0.0%),0 (0.0%),"1,411 (48.1%)",0 (0.0%),0 (0.0%),0 (0.0%)
F13,S-curve positive,"10,440 (100.0%)",0 (0.0%),"1,671 (16.0%)",0 (0.0%),0 (0.0%),0 (0.0%),"1,487 (14.2%)","3,292 (31.5%)",1 (0.0%),"3,989 (38.2%)"
F15,Threshold positive,"10,436 (100.0%)","2,172 (20.8%)","4,147 (39.7%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),"4,117 (39.4%)"
F18,Quadratic valley (U),"9,992 (100.0%)","4,318 (43.2%)",0 (0.0%),0 (0.0%),107 (1.1%),"3,212 (32.1%)",0 (0.0%),"2,345 (23.5%)",0 (0.0%),10 (0.1%)
F21,Cubic,"6,578 (100.0%)","3,479 (52.9%)","1,482 (22.5%)",845 (12.8%),37 (0.6%),0 (0.0%),0 (0.0%),735 (11.2%),0 (0.0%),0 (0.0%)
F22,Oscillation,"10,440 (100.0%)","10,440 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


## Diagnostic — misclassified F13 S-curve cases

This block reports every F13 case that entered the Simple path but was not labelled `S/threshold`. It also plots representative cases from each erroneous destination so that power-law truncation and spurious cubic turning points can be inspected directly.

In [ ]:
import matplotlib.pyplot as plt

F13_EXPECTED_BROAD = 'S/threshold'
F13_EXAMPLES_PER_DESTINATION = 8

f13_simple = data[(data['family_id'] == 'F13') & data['gate2'].eq('Simple')].copy()
f13_misclassified = f13_simple[~f13_simple['broad_shape'].eq(F13_EXPECTED_BROAD)].copy()

def _decode_shape_details(value):
    try:
        return json.loads(value) if isinstance(value, str) else (value or {})
    except (TypeError, json.JSONDecodeError):
        return {}

decoded = f13_misclassified['shape_details'].map(_decode_shape_details)
f13_misclassified['fit_step'] = decoded.map(lambda d: d.get('step'))
f13_misclassified['power_b'] = decoded.map(lambda d: d.get('b', np.nan))
f13_misclassified['fit_r2'] = decoded.map(lambda d: d.get('r2', np.nan))
f13_misclassified['n_turning_points'] = decoded.map(lambda d: d.get('n_tp', np.nan))

destination_counts = f13_misclassified.groupby(
    ['broad_shape', 'shape_label'], as_index=False
).size().rename(columns={'size': 'count'})
destination_counts['proportion_of_F13_simple'] = (
    destination_counts['count'] / len(f13_simple)
)
destination_counts['Count (proportion)'] = [
    count_percent(n, len(f13_simple)) for n in destination_counts['count']
]

print('=== Misclassified F13 S-curve cases ===')
print(f'F13 Simple input: {len(f13_simple):,}')
print(f'Misclassified: {count_percent(len(f13_misclassified), len(f13_simple))}')
display(destination_counts[['broad_shape', 'shape_label', 'Count (proportion)']])

diagnostic_columns = [
    'case_id', '_pt_idx', 'strength_name', 'snr_float', 'MIC', 'pearson_r',
    'spearman_rho', 'broad_shape', 'shape_label', 'fit_step', 'power_b',
    'fit_r2', 'n_turning_points',
]
diagnostic_columns = [c for c in diagnostic_columns if c in f13_misclassified.columns]
f13_diagnostic_table = f13_misclassified[diagnostic_columns].rename(
    columns={'_pt_idx': 'scatter_point_index'}
).sort_values(['broad_shape', 'snr_float', 'case_id'], ascending=[True, False, True])
f13_diagnostic_table.to_csv(OUT_DIR / 'misclassified_F13_cases.csv', index=False)
print('\nFirst 30 misclassified cases (the complete table is saved as CSV):')
display(f13_diagnostic_table.head(30))

# Select cases deterministically across the observed SNR range for each wrong destination.
representative_indices = []
for destination, group in f13_misclassified.groupby('broad_shape', sort=True):
    group = group.sort_values(['snr_float', 'case_id']).copy()
    n_take = min(F13_EXAMPLES_PER_DESTINATION, len(group))
    positions = np.unique(np.linspace(0, len(group) - 1, n_take).round().astype(int))
    representative_indices.extend(group.iloc[positions].index.tolist())

representatives = f13_misclassified.loc[representative_indices].sort_values(
    ['broad_shape', 'snr_float', 'case_id'], ascending=[True, False, True]
)
n_cols = 4
n_rows = max(1, math.ceil(len(representatives) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.6 * n_rows), squeeze=False)

for ax, (_, row) in zip(axes.ravel(), representatives.iterrows()):
    point_index = int(row['_pt_idx'])
    x = x_all[point_index].astype(float)
    y = y_all[point_index].astype(float)
    valid = np.isfinite(x) & np.isfinite(y)
    ax.scatter(x[valid], y[valid], s=8, alpha=0.30, color='#4c78a8', edgecolors='none')

    details = _decode_shape_details(row['shape_details'])
    grid = np.linspace(max(1e-6, np.nanmin(x[valid])), np.nanmax(x[valid]), 300)
    if details.get('step') == 'power_law':
        fitted = _power_law(grid, details['a'], details['b'], details['c'])
        ax.plot(grid, fitted, color='#e45756', lw=2)
        method_text = f"power: b={details['b']:.3f}, R²={details['r2']:.3f}"
    elif details.get('step') == 'cubic':
        coeffs = np.asarray(details['coeffs'], dtype=float)
        ax.plot(grid, np.polyval(coeffs, grid), color='#e45756', lw=2)
        roots = np.roots(np.polyder(coeffs))
        interior = np.sort(roots[np.isreal(roots)].real)
        interior = interior[(interior >= 0.05) & (interior <= 0.95)]
        for root in interior:
            ax.axvline(root, color='#f2cf5b', lw=1.5, ls='--')
        method_text = f"cubic: TP={len(interior)}, R²={details['r2']:.3f}"
    else:
        method_text = str(details.get('step', 'unknown'))

    snr_val = row.get('snr_float', row.get('actual_snr', '?'))
    ax.set_title(
        f"{row['case_id']} → {row['broad_shape']}\n"
        f"MIC={row['MIC']:.3f}, SNR={snr_val}; {method_text}",
        fontsize=9,
    )
    ax.grid(alpha=0.15)

for ax in axes.ravel()[len(representatives):]:
    ax.axis('off')

fig.suptitle('Representative misclassified F13 S-curve cases', fontsize=16, y=1.01)
fig.tight_layout()
diagnostic_figure_path = OUT_DIR / 'misclassified_F13_examples.png'
fig.savefig(diagnostic_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved complete case table: {OUT_DIR / "misclassified_F13_cases.csv"}')
print(f'Saved representative plot: {diagnostic_figure_path}')